### Synchronised Tracking

Tracks bottles across multiple videos and assigns a global ID to each physical bottle.

**Key features:**
- Uses the fine-tuned YOLO11n model from Notebook 5 (trained on your CVAT data).
- Processes the optimized videos in `videos_2/`.
- Uses a custom `GlobalRegistry` to match detections across videos based on their bounding box size and position.
- Ensures IDs are monotonically increasing.
- Caps the demo output to the first 60 seconds of each video.
- Generates one demo video per source video with the global IDs drawn on screen.

Outputs: annotated demo videos in `results/synchronised_tracking/demos/`.

In [1]:
from pathlib import Path
import time
import cv2
import torch
import numpy as np
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

VIDEO_DIR = ROOT / "videos_2"
OUTPUT_DIR = ROOT / "results" / "synchronised_tracking"
DEMO_DIR = OUTPUT_DIR / "demos"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

# Load the model trained in Notebook 5
MODEL_PATH = ROOT / "results" / "cvat_training" / "yolo11n_cvat_all" / "weights" / "best.pt"
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}. Run Notebook 5 first!")

model = YOLO(str(MODEL_PATH))
print(f"Loaded model: {MODEL_PATH.name}")
print(f"Classes: {model.names}")

PyTorch: 2.14.1+cpu
Using device: cpu
Loaded model: best.pt
Classes: {0: 'gas-bottles'}


### The Global Registry

When ByteTrack processes a video, it gives every bottle a local ID (1, 2, 3...). When you start a new video, ByteTrack resets and starts from 1 again.

To fix this, we wrap ByteTrack in a `GlobalRegistry`. 
- It keeps a list of every bottle it has ever seen.
- When a new bottle appears, it compares the bounding box size and center position. 
- If it matches an existing bottle, it assigns the **old** ID. 
- If it's a new bottle, it assigns the **next** ID (monotonically increasing).

In [2]:
class GlobalRegistry:
    def __init__(self, match_threshold=0.3, ttl_seconds=10):
        self.next_id = 1
        self.records = {}
        self.match_threshold = match_threshold
        self.ttl_seconds = ttl_seconds

    def _signature(self, box):
        x1, y1, x2, y2 = box
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        w = x2 - x1
        h = y2 - y1
        # Normalize based on expected bottle aspect ratio/size
        return np.array([cx, cy, w, h], dtype=float)

    def _distance(self, a, b):
        # Calculate distance relative to the size of the box
        # This ensures a moving bottle is still matched
        return float(np.linalg.norm(a - b) / (np.linalg.norm(a[2:]) + 1e-6))

    def assign(self, signature, timestamp):
        best_id = None
        best_score = 1e9

        for gid, rec in self.records.items():
            if timestamp - rec["last_seen"] > self.ttl_seconds:
                continue
            d = self._distance(signature, rec["signature"])
            if d < best_score:
                best_score = d
                best_id = gid

        # If we found a good match, update it and return the existing ID
        if best_id is not None and best_score < self.match_threshold:
            self.records[best_id]["signature"] = signature
            self.records[best_id]["last_seen"] = timestamp
            return best_id

        # Otherwise, create a new ID
        new_id = self.next_id
        self.next_id += 1
        self.records[new_id] = {
            "signature": signature,
            "first_seen": timestamp,
            "last_seen": timestamp,
        }
        return new_id

### Process Videos and Render Demos

For each video, we run tracking, pass the bounding boxes to the registry, and draw the global ID on the screen. We write the output to a new `.mp4` file.

We limit `max_frames` to `fps * 60`, so the output demo is capped at exactly 1 minute.

In [3]:
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}
video_paths = sorted(p for p in VIDEO_DIR.glob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)

if not video_paths:
    print(f"No videos found in {VIDEO_DIR}")
else:
    print(f"Found {len(video_paths)} videos to process.")
    
    # We use one shared registry across ALL videos so IDs stay consistent
    global_registry = GlobalRegistry(match_threshold=0.25, ttl_seconds=15)
    global_start_time = time.time()

    for video_path in video_paths:
        print(f"\nProcessing: {video_path.name}")
        
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        
        # Limit to 60 seconds per video for the demo
        max_frames = int(fps * 60)
        
        out_path = DEMO_DIR / f"sync_{video_path.stem}.mp4"
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))

        frame_count = 0
        unique_ids_in_video = set()

        # Use stream=True for memory efficiency
        results = model.track(
            source=str(video_path),
            tracker="bytetrack.yaml",
            conf=0.25,
            device=device,
            persist=True,
            stream=True,
            verbose=False
        )

        for r in results:
            if frame_count >= max_frames:
                break

            frame = r.orig_img.copy()
            boxes = r.boxes

            if boxes is not None and len(boxes) > 0:
                xyxy = boxes.xyxy.cpu().numpy()
                for box in xyxy:
                    # Get signature and assign global ID
                    sig = global_registry._signature(box)
                    gid = global_registry.assign(sig, time.time() - global_start_time)
                    unique_ids_in_video.add(gid)

                    # Draw box and text
                    x1, y1, x2, y2 = box.astype(int)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"Bottle ID: {gid}", (x1, max(y1 - 10, 20)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            writer.write(frame)
            frame_count += 1

        writer.release()
        
        print(f"  -> Saved {out_path.name} ({frame_count} frames)")
        print(f"  -> Global IDs seen in this video: {sorted(list(unique_ids_in_video))}")

    print("\nAll demos generated successfully.")
    print(f"Check the folder: {DEMO_DIR.relative_to(ROOT)}")

Found 6 videos to process.

Processing: optimized_14_43_back_left_cropped.mp4
  -> Saved sync_optimized_14_43_back_left_cropped.mp4 (1380 frames)
  -> Global IDs seen in this video: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Processing: optimized_14_43_back_right_cropped.mp4
  -> Saved sync_optimized_14_43_back_right_cropped.mp4 (1380 frames)
  -> Global IDs seen in this video: [19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]

Processing: optimized_14_43_front_cropped.mp4
  -> Saved sync_optimized_14_43_front_cropped.mp4 (1380 frames)
  -> Global IDs seen in this video: [28, 33, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]

Processing: optimized_14_55_back_left_cropped.mp4
  -> Saved sync_optimized_14_55_back_left_cropped.mp4 (1380 frames)
  -> Global IDs seen in this video: [47, 52, 57, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76]

Processing: optimized_14_55_bac

### Summary Statistics

In [5]:
print("=" * 60)
print("SYNCHRONISED TRACKING SUMMARY")
print("=" * 60)

if not video_paths:
    print("No videos were processed.")
else:
    print(f"Total videos processed: {len(video_paths)}")
    
    # Gather all IDs from the global registry
    all_ids = sorted(list(global_registry.records.keys()))
    print(f"Total unique bottles tracked across ALL videos: {len(all_ids)}")
    print(f"Global IDs issued: {all_ids}")
    
    print("\nPer-video breakdown:")
    print(f"{'Video Name':<35} | {'Frames Processed':<15} | {'IDs Seen'}")
    print("-" * 75)
    
    # Re-scan the videos quickly just to get the frame counts and IDs 
    # (Alternatively, you could have saved this in a list during the main loop)
    for video_path in video_paths:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        max_frames = int(fps * 60) # 1 minute cap
        total_frames_in_video = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        processed = min(max_frames, total_frames_in_video)
        cap.release()
        
        # We don't have the exact IDs per video saved in memory from the last cell, 
        # so we just print the total frames processed.
        print(f"{video_path.name:<35} | {processed:<15} | (See video output)")
        
    print("\n" + "=" * 60)
    print("OUTPUTS FOR REPORT")
    print("=" * 60)
    print(f"Demo videos saved to: {DEMO_DIR.relative_to(ROOT)}")

SYNCHRONISED TRACKING SUMMARY
Total videos processed: 6
Total unique bottles tracked across ALL videos: 116
Global IDs issued: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116]

Per-video breakdown:
Video Name                          | Frames Processed | IDs Seen
---------------------------------------------------------------------------
optimized_14_43_back_left_cropped.mp4 | 1380            | (See video output)
optimized_14_43_back_right_cropped.mp4 | 1380            | (See video output)
optimized_14_43_front_cropped.mp4   | 1380            | (See video output)
optimi